In [3]:
import requests
import folium
from folium import plugins
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import os

# ==========================================
# 1. DATASET - 31 LOKASI (SURABAYA)
# ==========================================
locations = {
    "1. Monumen Kapal Selam": {"coord": (-7.2654, 112.7503), "elev": 4.0},
    "2. Tugu Pahlawan": {"coord": (-7.2453, 112.7379), "elev": 9.0},
    "3. Jembatan Merah": {"coord": (-7.2365, 112.7383), "elev": 6.0},
    "4. Hotel Majapahit": {"coord": (-7.2573, 112.7388), "elev": 9.0},
    "5. Balai Kota Surabaya": {"coord": (-7.2589, 112.7469), "elev": 6.0},
    "6. House of Sampoerna": {"coord": (-7.2306, 112.7343), "elev": 6.0},
    "7. Museum Surabaya (Siola)": {"coord": (-7.2514, 112.7363), "elev": 7.0},
    "8. Rumah WR Soepratman": {"coord": (-7.2504, 112.7538), "elev": 5.0},
    "9. Gedung De Javasche Bank": {"coord": (-7.2365, 112.7358), "elev": 11.0},
    "10. Gereja Katolik Kepanjen": {"coord": (-7.2435, 112.7335), "elev": 6.0},
    "11. Jembatan Petekan": {"coord": (-7.222192, 112.738044), "elev": 5.0},
    "12. Gedung Internatio": {"coord": (-7.236281, 112.736915), "elev": 11.0},
    "13. Gedung Negara Grahadi": {"coord": (-7.263524, 112.743179), "elev": 7.0},
    "14. Kantor Pos Kebon Rojo": {"coord": (-7.243218, 112.737700), "elev": 6.0},
    "15. Rumah HOS Tjokroaminoto": {"coord": (-7.252464, 112.737747), "elev": 7.0},
    "16. Makam Belanda Peneleh": {"coord": (-7.253019586881285, 112.74035309946457), "elev": 5.0},
    "17. Kampung Lawang Seketeng": {"coord": (-7.250495592744265, 112.74075173450092), "elev": 7.0},
    "18. Gerbang Depan ITS": {"coord": (-7.279395032652996, 112.79009468032517), "elev": 3.0},
    "19. Gedung Cerutu": {"coord": (-7.2360970729302325, 112.73704713519352), "elev": 11.0},
    "20. Patung Karapan Sapi": {"coord": (-7.272209, 112.742095), "elev": 26.0},
    "21. Klenteng Sanggar Agung": {"coord": (-7.247199578801502, 112.80219558900477), "elev": 0.0},
    "22. Museum Pendidikan": {"coord": (-7.255254406638845, 112.74278515368215), "elev": 6.0},
    "23. Klenteng Hong Tiek Hian": {"coord": (-7.2367971558297395, 112.74387553040685), "elev": 5.0},
    "24. Monumen Bambu Runcing": {"coord": (-7.267096754354455, 112.74418425368475), "elev": 7.0},
    "25. Taman Prestasi": {"coord": (-7.261174085645066, 112.74291416110384), "elev": 5.0},
    "26. Penjara Kalisosok": {"coord": (-7.2343, 112.7351), "elev": 5.0},
    "27. Masjid Nasional Al-Akbar": {"coord": (-7.3381, 112.7148), "elev": 12.0},
    "28. Jembatan Suroboyo (Kenjeran)": {"coord": (-7.2515, 112.7964), "elev": 2.0},
    "29. Monumen Jenderal Sudirman": {"coord": (-7.2736, 112.7441), "elev": 8.0},
    "30. Kawasan Kota Tua Kembang Jepun": {"coord": (-7.2384, 112.7412), "elev": 6.0},
    "31. Pura Agung Jagat Karana": {"coord": (-7.2325, 112.7291), "elev": 5.0}
}

names = list(locations.keys())
coords_values = [loc["coord"] for loc in locations.values()]
elevations = {name: loc["elev"] for name, loc in locations.items()}
elevations_list = [loc["elev"] for loc in locations.values()]
n = len(locations)

START_CITY = "18. Gerbang Depan ITS"
start_location_idx = names.index(START_CITY)

print("=" * 100)
print("🚴 GENETIC ALGORITHM - CYCLING ROUTE OPTIMIZATION (FATIGUE-AWARE)")
print("OBJECTIVE: shortest distance + minimize max fatigue + avoid extreme climbs")
print("=" * 100)

# ==========================================
# 2. FETCH DISTANCE MATRIX - SEPEDA (OSRM)
# ==========================================
OUT_DIR = "hasil_ga_fatigue_31"
os.makedirs(OUT_DIR, exist_ok=True)

cache_path = os.path.join(OUT_DIR, "distance_matrix_km.npy")

if os.path.exists(cache_path):
    print("\n💾 Load distance matrix dari cache...\n")
    distance_matrix = np.load(cache_path)
else:
    print("\n📡 Mengambil Distance Matrix untuk SEPEDA dari OSRM...\n")
    coords_list = [f"{lon},{lat}" for lat, lon in coords_values]
    coords_string = ";".join(coords_list)
    url = f"http://router.project-osrm.org/table/v1/bike/{coords_string}?annotations=distance"

    try:
        response = requests.get(url, timeout=60).json()
        if response.get("code") == "Ok":
            distance_matrix = np.array(response["distances"]) / 1000.0
            np.save(cache_path, distance_matrix)
            print(f"✅ Distance Matrix SEPEDA OK! (cached: {cache_path})\n")
        else:
            print(f"❌ OSRM Error: {response}")
            raise SystemExit(1)
    except Exception as e:
        print(f"❌ OSRM Error: {e}")
        raise SystemExit(1)

# ==========================================
# 3. CONSTRAINT / PENALTY SETUP (IKUT TEMEN ACO)
# ==========================================
LAMBDA_FATIGUE = 2.5
MU_VIOLATIONS = 10.0
RECOVERY_FACTOR = 0.82
EXTREME_EFFORT_M = 0.5  # naik > 5m antar titik dianggap ekstrem

def get_metrics(route):
    """
    route: list of indices (GA creates a cycle, last->first)
    Return: score, dist_km, max_fatigue, violations, fatigue_history
    """
    dist = 0.0
    fatigue = 0.0
    max_fatigue = 0.0
    violations = 0
    f_history = [0.0]

    for i in range(len(route)):
        c = route[i]
        nxt = route[(i + 1) % len(route)]

        dist += distance_matrix[c][nxt]

        elev_diff = elevations_list[nxt] - elevations_list[c]
        effort = max(0.0, elev_diff)

        if effort > EXTREME_EFFORT_M:
            violations += 1

        fatigue = (fatigue + effort) * RECOVERY_FACTOR
        max_fatigue = max(max_fatigue, fatigue)
        f_history.append(fatigue)

    score = dist + (LAMBDA_FATIGUE * max_fatigue) + (MU_VIOLATIONS * violations)
    return score, dist, max_fatigue, violations, f_history

# ==========================================
# 4. GA CLASS (start fixed at ITS)
# ==========================================
class GeneticAlgorithmFatigueTSP:
    def __init__(self, pop_size=120, max_gen=350, patience=60, min_improvement=0.0002, run_name="Run"):
        self.pop_size = pop_size
        self.max_gen = max_gen
        self.patience = patience
        self.min_improvement = min_improvement
        self.run_name = run_name

        self.best_route = None
        self.best_score = float("inf")
        self.score_history = []
        self.stopped_early = False
        self.stopped_at_gen = 0

    def create_individual(self):
        other = [i for i in range(n) if i != start_location_idx]
        random.shuffle(other)
        return [start_location_idx] + other  # cycle assumed

    def create_population(self):
        return [self.create_individual() for _ in range(self.pop_size)]

    def fitness(self, route):
        score, *_ = get_metrics(route)
        return 1.0 / (1.0 + score)

    def selection(self, pop, fitness_scores, k=5):
        idxs = random.sample(range(len(pop)), k)
        best = max(idxs, key=lambda i: fitness_scores[i])
        return pop[best].copy()

    def crossover(self, p1, p2):
        # OX-like crossover, keep start fixed at index 0
        size = len(p1)
        start = random.randint(1, size - 2)
        end = random.randint(start, size - 1)

        child = [None] * size
        child[0] = start_location_idx
        child[start:end + 1] = p1[start:end + 1]

        ptr = 1
        for gene in p2[1:] + p2[1:]:
            if gene not in child:
                while ptr < size and child[ptr] is not None:
                    ptr += 1
                if ptr >= size:
                    break
                child[ptr] = gene

        # fill remaining
        for gene in range(n):
            if gene not in child:
                for i in range(1, size):
                    if child[i] is None:
                        child[i] = gene
                        break
        return child

    def mutate(self, route, mutation_rate=0.20):
        if random.random() < mutation_rate:
            i, j = random.sample(range(1, len(route)), 2)
            route[i], route[j] = route[j], route[i]
        return route

    def evolve(self, verbose=False):
        if verbose:
            print(f"🧬 {self.run_name}...\n")

        pop = self.create_population()
        no_improvement_count = 0
        last_best = float("inf")

        for gen in range(self.max_gen):
            fitness_scores = [self.fitness(ind) for ind in pop]
            best_idx = int(np.argmax(fitness_scores))
            best_route_gen = pop[best_idx].copy()

            best_score_gen, best_dist_gen, best_maxf_gen, best_viol_gen, _ = get_metrics(best_route_gen)
            self.score_history.append(best_score_gen)

            if best_score_gen < self.best_score:
                self.best_score = best_score_gen
                self.best_route = best_route_gen.copy()

            improvement = (last_best - best_score_gen) / (last_best + 1e-12)
            if improvement > self.min_improvement:
                no_improvement_count = 0
                last_best = best_score_gen
            else:
                no_improvement_count += 1

            if verbose and (gen + 1) % 50 == 0:
                print(f"  Gen {gen+1:3d}: score={best_score_gen:.2f} | dist={best_dist_gen:.2f}km | maxF={best_maxf_gen:.2f} | viol={best_viol_gen}")

            if no_improvement_count >= self.patience:
                self.stopped_early = True
                self.stopped_at_gen = gen + 1
                if verbose:
                    print(f"🛑 Early stop at gen {self.stopped_at_gen}")
                break

            # elitism
            elite_size = min(15, self.pop_size)
            elite_idx = np.argsort(fitness_scores)[-elite_size:]
            elite = [pop[i].copy() for i in elite_idx]

            offspring = []
            while len(offspring) < (self.pop_size - elite_size):
                p1 = self.selection(pop, fitness_scores, k=5)
                p2 = self.selection(pop, fitness_scores, k=5)
                child = self.crossover(p1, p2)
                child = self.mutate(child, mutation_rate=0.20)
                offspring.append(child)

            pop = elite + offspring

        if not self.stopped_early:
            self.stopped_at_gen = self.max_gen

        return self.best_route, self.best_score, self.score_history

# ==========================================
# 5. RUN GA (MULTI RUN)
# ==========================================
NUM_RUNS = 5  # dipercepat biar ga lama
print(f"\n🚀 Running {NUM_RUNS} GA iterations (SEPEDA)...\n")
print("=" * 100)

all_results = []
best_overall_route = None
best_overall_score = float("inf")

for run_num in range(1, NUM_RUNS + 1):
    print(f"\n📊 RUN {run_num}/{NUM_RUNS}")

    ga = GeneticAlgorithmFatigueTSP(
        pop_size=120,
        max_gen=350,
        patience=60,
        min_improvement=0.0002,
        run_name=f"Run {run_num}"
    )

    best_route, score_history = ga.evolve(verbose=True)[:2]  # (route, score, history) but we only need route+score
    best_route = ga.best_route
    best_score = ga.best_score

    score, dist_km, max_f, viol, f_hist = get_metrics(best_route)
    time_minutes = dist_km / 20 * 60

    print(f"✅ Score={score:.2f} | Dist={dist_km:.2f}km | MaxFatigue={max_f:.2f} | Viol={viol} | Gen={ga.stopped_at_gen}")

    all_results.append({
        "Run": run_num,
        "Score": score,
        "Dist_km": dist_km,
        "MaxFatigue": max_f,
        "Violations": viol,
        "Generations": ga.stopped_at_gen,
        "Route": best_route.copy(),
        "ScoreHistory": ga.score_history.copy()
    })

    if score < best_overall_score:
        best_overall_score = score
        best_overall_route = best_route.copy()

# export summary
df = pd.DataFrame([{
    "Run": r["Run"],
    "Score": r["Score"],
    "Dist_km": r["Dist_km"],
    "MaxFatigue": r["MaxFatigue"],
    "Violations": r["Violations"],
    "Generations": r["Generations"]
} for r in all_results])

summary_csv = os.path.join(OUT_DIR, "summary_results_ga.csv")
df.to_csv(summary_csv, index=False, encoding="utf-8")
print(f"\n✓ {summary_csv}")
print("\n" + df.to_string(index=False))

# ==========================================
# 6. PLOTS (RINGKAS & RELEVAN)
# ==========================================
# 6.1 Score per run
plt.figure(figsize=(10, 5))
plt.plot(df["Run"], df["Score"], marker="o", linewidth=2)
plt.title("GA Fatigue-Aware Score per Run (Lower is Better)")
plt.xlabel("Run")
plt.ylabel("Score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "score_per_run.png"), dpi=300)
plt.close()

# 6.2 Fatigue profile best route
best_score, best_dist, best_maxf, best_viol, best_f_hist = get_metrics(best_overall_route)
plt.figure(figsize=(12, 5))
plt.plot(best_f_hist, marker="o", linewidth=2, color="#EA4335")
plt.fill_between(range(len(best_f_hist)), best_f_hist, alpha=0.15, color="#EA4335")
plt.title("Fatigue Profile (Best Route)")
plt.xlabel("Step")
plt.ylabel("Fatigue")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fatigue_profile_best.png"), dpi=300)
plt.close()

# 6.3 Convergence (ambil history dari best run)
best_run = min(all_results, key=lambda r: r["Score"])
plt.figure(figsize=(10, 5))
plt.plot(best_run["ScoreHistory"], linewidth=2, color="#005aab")
plt.title("Convergence (Best Run) - Score vs Generation")
plt.xlabel("Generation")
plt.ylabel("Score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "convergence_best_run.png"), dpi=300)
plt.close()

print(f"✓ {os.path.join(OUT_DIR, 'score_per_run.png')}")
print(f"✓ {os.path.join(OUT_DIR, 'fatigue_profile_best.png')}")
print(f"✓ {os.path.join(OUT_DIR, 'convergence_best_run.png')}")

# ==========================================
# 7. CREATE INTERACTIVE MAP - SEPEDA (FOLIUM)
# ==========================================
print("\n🗺️ Creating interactive map (SEPEDA)...\n")

best_route = best_overall_route
score, dist_km, max_f, viol, f_hist = get_metrics(best_route)

best_time_minutes = dist_km / 20 * 60
hours = int(best_time_minutes // 60)
mins = int(best_time_minutes % 60)

center_lat = float(np.mean([c[0] for c in coords_values]))
center_lon = float(np.mean([c[1] for c in coords_values]))

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='CartoDB positron')

# OSRM route geometry
route_indices = best_route + [best_route[0]]
route_coords_ordered = [f"{coords_values[i][1]},{coords_values[i][0]}" for i in route_indices]
url_route = f"http://router.project-osrm.org/route/v1/bike/{';'.join(route_coords_ordered)}?overview=full&geometries=geojson"

print("📍 Fetching bike route from OSRM...")
try:
    res = requests.get(url_route, timeout=60).json()
    if res.get("code") == "Ok":
        geom = [(lat, lon) for lon, lat in res["routes"][0]["geometry"]["coordinates"]]

        plugins.AntPath(
            locations=geom,
            dash_array=[10, 20],
            delay=800,
            color='#005aab',
            pulse_color='white',
            weight=5,
            opacity=0.9,
        ).add_to(m)

        print("✓ Bike route OK!")
    else:
        print(f"⚠ OSRM route error: {res}")
except Exception as e:
    print(f"⚠ Route error: {e}")

# markers + popup
print("📌 Adding markers...")
for idx, route_idx in enumerate(best_route):
    name = names[route_idx]
    coord = coords_values[route_idx]
    elev = elevations[name]

    next_idx = best_route[(idx + 1) % len(best_route)]
    segment_dist = distance_matrix[route_idx][next_idx]
    elev_diff = elevations_list[next_idx] - elevations_list[route_idx]
    effort = max(0.0, elev_diff)

    marker_text = " ⭐ START/END" if route_idx == start_location_idx else ""
    icon_color = "green" if route_idx == start_location_idx else "blue"

    popup_html = f"""
    <div style="font-family: Arial; width: 340px;">
        <b style="color: #005aab;">🚴 STOP {idx + 1}/{len(best_route)}</b><br>
        <b>{name}{marker_text}</b><br>
        <hr style="margin: 5px 0;">
        <b>🏔️ Elevation: {elev:.1f} m</b><br>
        <b>📏 Distance to next: {segment_dist:.2f} km</b><br>
        <b>⬆ Effort (climb): {effort:.1f} m</b>
    </div>
    """

    folium.Marker(
        location=coord,
        popup=folium.Popup(popup_html, max_width=340),
        icon=folium.Icon(color=icon_color, icon='bicycle', prefix='fa')
    ).add_to(m)

    folium.CircleMarker(
        location=coord,
        radius=7,
        color=icon_color,
        fill=True,
        fillColor=icon_color,
        fillOpacity=0.2,
        weight=2
    ).add_to(m)

# overlay info
info_html = f"""
<div style="position: fixed; top: 20px; left: 70px; width: 520px;
            background-color: white; border-radius: 12px; z-index: 9999;
            font-family: 'Segoe UI'; box-shadow: 0 4px 16px rgba(0,0,0,0.15);">
<div style="display: flex; gap: 12px; padding: 20px; border-bottom: 1px solid #e0e0e0;
            background: linear-gradient(135deg, #005aab 0%, #003a70 100%);">
    <div style="font-size: 40px;">🚴</div>
    <div>
        <div style="font-size: 20px; font-weight: 700; color: white;">GA FATIGUE-AWARE ROUTE</div>
        <div style="font-size: 12px; color: #ddd;">dist + λ·max_fatigue + μ·violations</div>
    </div>
</div>

<div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 0;">
    <div style="padding: 14px; text-align: center; border-right: 1px solid #f0f0f0;">
        <div style="font-size: 11px; color: #757575;">📍 JARAK</div>
        <div style="font-size: 18px; font-weight: 700;">{dist_km:.2f} km</div>
    </div>
    <div style="padding: 14px; text-align: center; border-right: 1px solid #f0f0f0;">
        <div style="font-size: 11px; color: #757575;">🔥 MAX FATIGUE</div>
        <div style="font-size: 18px; font-weight: 700; color: #FFA500;">{max_f:.2f}</div>
    </div>
    <div style="padding: 14px; text-align: center;">
        <div style="font-size: 11px; color: #757575;">⛔ VIOLATIONS</div>
        <div style="font-size: 18px; font-weight: 700; color: #EA4335;">{viol}</div>
    </div>
</div>

<div style="padding: 14px 20px; background-color: #f5f5f5; text-align: center; border-bottom: 1px solid #e0e0e0;">
    <div style="font-size: 12px;">🕐 WAKTU (20 km/h - SEPEDA)</div>
    <div style="font-size: 20px; font-weight: 700;">{best_time_minutes:.1f} min ({hours}h {mins}m)</div>
</div>

<div style="padding: 12px 20px; font-size: 12px; line-height: 1.8;">
    <div>🎯 Start/End: {START_CITY.split('. ')[1]}</div>
    <div>📍 Total Stops: {len(best_route)}</div>
    <div>⚙️ Algorithm: Genetic Algorithm (GA)</div>
    <div style="margin-top: 8px; padding-top: 8px; border-top: 1px solid #e0e0e0; color: #666;">
        λ={LAMBDA_FATIGUE}, μ={MU_VIOLATIONS}, r={RECOVERY_FACTOR}, extreme>{EXTREME_EFFORT_M:.0f}m
    </div>
</div>
</div>
"""

m.get_root().html.add_child(folium.Element(info_html))

out_html = os.path.join(OUT_DIR, 'optimal_route_ga_fatigue_31.html')
m.save(out_html)
print(f"✓ {out_html}")

# ==========================================
# 8. EXPORT ROUTE DETAIL CSV
# ==========================================
route_detail = []
fatigue = 0.0
for idx in range(len(best_route)):
    cur = best_route[idx]
    nxt = best_route[(idx + 1) % len(best_route)]

    seg_dist = distance_matrix[cur][nxt]
    elev_diff = elevations_list[nxt] - elevations_list[cur]
    effort = max(0.0, elev_diff)
    violation = 1 if effort > EXTREME_EFFORT_M else 0

    fatigue = (fatigue + effort) * RECOVERY_FACTOR

    marker = " ⭐ START/END" if cur == start_location_idx else ""
    route_detail.append({
        "Stop": idx + 1,
        "Location": names[cur] + marker,
        "Elevation_m": elevations_list[cur],
        "Dist_to_next_km": round(seg_dist, 3),
        "Effort_climb_m": round(effort, 1),
        "Violation": violation,
        "Fatigue_after_leg": round(fatigue, 3)
    })

df_route = pd.DataFrame(route_detail)
route_csv = os.path.join(OUT_DIR, "route_detail_ga_fatigue_31.csv")
df_route.to_csv(route_csv, index=False, encoding="utf-8")
print(f"✓ {route_csv}")

print("\n" + "=" * 100)
print("✅ OPTIMIZATION COMPLETE - GA + FATIGUE-AWARE CONSTRAINTS (31 lokasi)")
print("=" * 100)
print(f"""
BEST RESULT:
  Score       : {best_score:.2f}
  Distance    : {best_dist:.2f} km
  Max Fatigue : {best_maxf:.2f}
  Violations  : {best_viol}

FILES:
  • {OUT_DIR}/optimal_route_ga_fatigue_31.html  (🗺️ peta folium untuk screenshot)
  • {OUT_DIR}/summary_results_ga.csv           (📋 ringkasan run)
  • {OUT_DIR}/route_detail_ga_fatigue_31.csv   (📋 detail rute)
  • {OUT_DIR}/score_per_run.png                (📈 skor per run)
  • {OUT_DIR}/fatigue_profile_best.png         (📈 fatigue terbaik)
  • {OUT_DIR}/convergence_best_run.png         (📈 konvergensi run terbaik)
""")

🚴 GENETIC ALGORITHM - CYCLING ROUTE OPTIMIZATION (FATIGUE-AWARE)
OBJECTIVE: shortest distance + minimize max fatigue + avoid extreme climbs

💾 Load distance matrix dari cache...


🚀 Running 5 GA iterations (SEPEDA)...


📊 RUN 1/5
🧬 Run 1...



C:\Users\asus\AppData\Local\Temp\ipykernel_26748\2101283160.py:218: RuntimeWarning: invalid value encountered in scalar divide
  improvement = (last_best - best_score_gen) / (last_best + 1e-12)


  Gen  50: score=201.06 | dist=114.02km | maxF=18.82 | viol=4
🛑 Early stop at gen 60
✅ Score=198.74 | Dist=111.70km | MaxFatigue=18.82 | Viol=4 | Gen=60

📊 RUN 2/5
🧬 Run 2...

  Gen  50: score=197.53 | dist=100.38km | maxF=18.86 | viol=5
🛑 Early stop at gen 60
✅ Score=192.34 | Dist=105.19km | MaxFatigue=18.86 | Viol=4 | Gen=60

📊 RUN 3/5
🧬 Run 3...

  Gen  50: score=193.10 | dist=95.95km | maxF=18.86 | viol=5
🛑 Early stop at gen 60
✅ Score=192.47 | Dist=95.32km | MaxFatigue=18.86 | Viol=5 | Gen=60

📊 RUN 4/5
🧬 Run 4...

  Gen  50: score=201.85 | dist=104.70km | maxF=18.86 | viol=5
🛑 Early stop at gen 60
✅ Score=200.76 | Dist=103.61km | MaxFatigue=18.86 | Viol=5 | Gen=60

📊 RUN 5/5
🧬 Run 5...

  Gen  50: score=186.25 | dist=99.42km | maxF=18.73 | viol=4
🛑 Early stop at gen 60
✅ Score=183.18 | Dist=96.35km | MaxFatigue=18.73 | Viol=4 | Gen=60

✓ hasil_ga_fatigue_31\summary_results_ga.csv

 Run      Score  Dist_km  MaxFatigue  Violations  Generations
   1 198.742514 111.6955   18.818806  

In [ ]:
import requests
import folium
from folium import plugins
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd
import os
import optuna

# ==========================================
# 1) DATASET - 31 LOKASI (SURABAYA)
# ==========================================
locations = {
    "1. Monumen Kapal Selam": {"coord": (-7.2654, 112.7503), "elev": 4.0},
    "2. Tugu Pahlawan": {"coord": (-7.2453, 112.7379), "elev": 9.0},
    "3. Jembatan Merah": {"coord": (-7.2365, 112.7383), "elev": 6.0},
    "4. Hotel Majapahit": {"coord": (-7.2573, 112.7388), "elev": 9.0},
    "5. Balai Kota Surabaya": {"coord": (-7.2589, 112.7469), "elev": 6.0},
    "6. House of Sampoerna": {"coord": (-7.2306, 112.7343), "elev": 6.0},
    "7. Museum Surabaya (Siola)": {"coord": (-7.2514, 112.7363), "elev": 7.0},
    "8. Rumah WR Soepratman": {"coord": (-7.2504, 112.7538), "elev": 5.0},
    "9. Gedung De Javasche Bank": {"coord": (-7.2365, 112.7358), "elev": 11.0},
    "10. Gereja Katolik Kepanjen": {"coord": (-7.2435, 112.7335), "elev": 6.0},
    "11. Jembatan Petekan": {"coord": (-7.222192, 112.738044), "elev": 5.0},
    "12. Gedung Internatio": {"coord": (-7.236281, 112.736915), "elev": 11.0},
    "13. Gedung Negara Grahadi": {"coord": (-7.263524, 112.743179), "elev": 7.0},
    "14. Kantor Pos Kebon Rojo": {"coord": (-7.243218, 112.737700), "elev": 6.0},
    "15. Rumah HOS Tjokroaminoto": {"coord": (-7.252464, 112.737747), "elev": 7.0},
    "16. Makam Belanda Peneleh": {"coord": (-7.253019586881285, 112.74035309946457), "elev": 5.0},
    "17. Kampung Lawang Seketeng": {"coord": (-7.250495592744265, 112.74075173450092), "elev": 7.0},
    "18. Gerbang Depan ITS": {"coord": (-7.279395032652996, 112.79009468032517), "elev": 3.0},
    "19. Gedung Cerutu": {"coord": (-7.2360970729302325, 112.73704713519352), "elev": 11.0},
    "20. Patung Karapan Sapi": {"coord": (-7.272209, 112.742095), "elev": 26.0},
    "21. Klenteng Sanggar Agung": {"coord": (-7.247199578801502, 112.80219558900477), "elev": 0.0},
    "22. Museum Pendidikan": {"coord": (-7.255254406638845, 112.74278515368215), "elev": 6.0},
    "23. Klenteng Hong Tiek Hian": {"coord": (-7.2367971558297395, 112.74387553040685), "elev": 5.0},
    "24. Monumen Bambu Runcing": {"coord": (-7.267096754354455, 112.74418425368475), "elev": 7.0},
    "25. Taman Prestasi": {"coord": (-7.261174085645066, 112.74291416110384), "elev": 5.0},
    "26. Penjara Kalisosok": {"coord": (-7.2343, 112.7351), "elev": 5.0},
    "27. Masjid Nasional Al-Akbar": {"coord": (-7.3381, 112.7148), "elev": 12.0},
    "28. Jembatan Suroboyo (Kenjeran)": {"coord": (-7.2515, 112.7964), "elev": 2.0},
    "29. Monumen Jenderal Sudirman": {"coord": (-7.2736, 112.7441), "elev": 8.0},
    "30. Kawasan Kota Tua Kembang Jepun": {"coord": (-7.2384, 112.7412), "elev": 6.0},
    "31. Pura Agung Jagat Karana": {"coord": (-7.2325, 112.7291), "elev": 5.0}
}

names = list(locations.keys())
coords_values = [loc["coord"] for loc in locations.values()]
elevations_list = [loc["elev"] for loc in locations.values()]
n = len(locations)

START_CITY = "18. Gerbang Depan ITS"
start_location_idx = names.index(START_CITY)

# ==========================================
# 2) OUTPUT FOLDER + OSRM MATRIX (CACHE)
# ==========================================
OUT_DIR = "hasil_ga_fatigue_optuna_31"
os.makedirs(OUT_DIR, exist_ok=True)

cache_path = os.path.join(OUT_DIR, "distance_matrix_km.npy")

if os.path.exists(cache_path):
    print("\n💾 Load distance matrix dari cache...\n")
    distance_matrix = np.load(cache_path)
else:
    print("\n📡 Mengambil Distance Matrix untuk SEPEDA dari OSRM...\n")
    coords_list = [f"{lon},{lat}" for lat, lon in coords_values]
    coords_string = ";".join(coords_list)
    url = f"http://router.project-osrm.org/table/v1/bike/{coords_string}?annotations=distance"
    response = requests.get(url, timeout=60).json()
    if response.get("code") != "Ok":
        raise RuntimeError(f"OSRM Error: {response}")
    distance_matrix = np.array(response["distances"]) / 1000.0
    np.save(cache_path, distance_matrix)
    print(f"✅ Distance Matrix OK! cached: {cache_path}")

# ==========================================
# 3) GLOBAL PARAMS (DI-TUNE OPTUNA)
# ==========================================
LAMBDA_FATIGUE = 2.5
MU_VIOLATIONS = 10.0
RECOVERY_FACTOR = 0.82
EXTREME_EFFORT_M = 10.0

# ==========================================
# 4) METRICS (IKUT TEMEN ACO)
# ==========================================
def get_metrics(route):
    dist = 0.0
    fatigue = 0.0
    max_fatigue = 0.0
    violations = 0
    f_history = [0.0]

    for i in range(len(route)):
        c = route[i]
        nxt = route[(i + 1) % len(route)]

        dist += distance_matrix[c][nxt]

        elev_diff = elevations_list[nxt] - elevations_list[c]
        effort = max(0.0, elev_diff)

        if effort > EXTREME_EFFORT_M:
            violations += 1

        fatigue = (fatigue + effort) * RECOVERY_FACTOR
        max_fatigue = max(max_fatigue, fatigue)
        f_history.append(fatigue)

    score = dist + (LAMBDA_FATIGUE * max_fatigue) + (MU_VIOLATIONS * violations)
    return score, dist, max_fatigue, violations, f_history

# ==========================================
# 5) GA CLASS
# ==========================================
class GeneticAlgorithmFatigueTSP:
    def __init__(self, pop_size=120, max_gen=350, patience=60, min_improvement=0.0002,
                 mutation_rate=0.20, elite_size=15, tournament_k=5, run_name="Run", seed=None):
        self.pop_size = pop_size
        self.max_gen = max_gen
        self.patience = patience
        self.min_improvement = min_improvement
        self.mutation_rate = mutation_rate
        self.elite_size = min(elite_size, pop_size)
        self.tournament_k = tournament_k
        self.run_name = run_name

        self.best_route = None
        self.best_score = float("inf")
        self.score_history = []
        self.stopped_early = False
        self.stopped_at_gen = 0

        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

    def create_individual(self):
        other = [i for i in range(n) if i != start_location_idx]
        random.shuffle(other)
        return [start_location_idx] + other

    def create_population(self):
        return [self.create_individual() for _ in range(self.pop_size)]

    def fitness(self, route):
        score, *_ = get_metrics(route)
        return 1.0 / (1.0 + score)

    def selection(self, pop, fitness_scores):
        idxs = random.sample(range(len(pop)), self.tournament_k)
        best = max(idxs, key=lambda i: fitness_scores[i])
        return pop[best].copy()

    def crossover(self, p1, p2):
        size = len(p1)
        start = random.randint(1, size - 2)
        end = random.randint(start, size - 1)

        child = [None] * size
        child[0] = start_location_idx
        child[start:end + 1] = p1[start:end + 1]

        ptr = 1
        for gene in p2[1:] + p2[1:]:
            if gene not in child:
                while ptr < size and child[ptr] is not None:
                    ptr += 1
                if ptr >= size:
                    break
                child[ptr] = gene

        for gene in range(n):
            if gene not in child:
                for i in range(1, size):
                    if child[i] is None:
                        child[i] = gene
                        break
        return child

    def mutate(self, route):
        if random.random() < self.mutation_rate:
            i, j = random.sample(range(1, len(route)), 2)
            route[i], route[j] = route[j], route[i]
        return route

    def evolve(self, verbose=False):
        if verbose:
            print(f"🧬 {self.run_name}...\n")

        pop = self.create_population()
        no_improvement_count = 0
        last_best = float("inf")

        for gen in range(self.max_gen):
            fitness_scores = [self.fitness(ind) for ind in pop]
            best_idx = int(np.argmax(fitness_scores))
            best_route_gen = pop[best_idx].copy()

            best_score_gen, best_dist_gen, best_maxf_gen, best_viol_gen, _ = get_metrics(best_route_gen)
            self.score_history.append(best_score_gen)

            if best_score_gen < self.best_score:
                self.best_score = best_score_gen
                self.best_route = best_route_gen.copy()

            improvement = (last_best - best_score_gen) / (last_best + 1e-12)
            if improvement > self.min_improvement:
                no_improvement_count = 0
                last_best = best_score_gen
            else:
                no_improvement_count += 1

            if verbose and (gen + 1) % 50 == 0:
                print(f"  Gen {gen+1:3d}: score={best_score_gen:.2f} | dist={best_dist_gen:.2f}km | maxF={best_maxf_gen:.2f} | viol={best_viol_gen}")

            if no_improvement_count >= self.patience:
                self.stopped_early = True
                self.stopped_at_gen = gen + 1
                if verbose:
                    print(f"🛑 Early stop at gen {self.stopped_at_gen}")
                break

            elite_idx = np.argsort(fitness_scores)[-self.elite_size:]
            elite = [pop[i].copy() for i in elite_idx]

            offspring = []
            while len(offspring) < (self.pop_size - self.elite_size):
                p1 = self.selection(pop, fitness_scores)
                p2 = self.selection(pop, fitness_scores)
                child = self.crossover(p1, p2)
                child = self.mutate(child)
                offspring.append(child)

            pop = elite + offspring

        if not self.stopped_early:
            self.stopped_at_gen = self.max_gen

        return self.best_route, self.best_score, self.score_history

# ==========================================
# 6) OPTUNA TUNING
# ==========================================
def run_ga_once(pop_size, max_gen, patience, min_improvement, mutation_rate, elite_size, tournament_k, seed=123):
    ga = GeneticAlgorithmFatigueTSP(
        pop_size=pop_size,
        max_gen=max_gen,
        patience=patience,
        min_improvement=min_improvement,
        mutation_rate=mutation_rate,
        elite_size=elite_size,
        tournament_k=tournament_k,
        run_name="OptunaTrial",
        seed=seed
    )
    ga.evolve(verbose=False)
    route = ga.best_route
    score, dist_km, max_f, viol, _ = get_metrics(route)
    return score, dist_km, max_f, viol

def optuna_objective(trial):
    global LAMBDA_FATIGUE, MU_VIOLATIONS, RECOVERY_FACTOR, EXTREME_EFFORT_M

    # constraint params
    LAMBDA_FATIGUE = trial.suggest_float("LAMBDA_FATIGUE", 0.5, 10.0, log=True)
    MU_VIOLATIONS = trial.suggest_float("MU_VIOLATIONS", 1.0, 80.0, log=True)
    RECOVERY_FACTOR = trial.suggest_float("RECOVERY_FACTOR", 0.70, 0.95)
    EXTREME_EFFORT_M = trial.suggest_float("EXTREME_EFFORT_M", 3.0, 15.0)

    # GA params
    pop_size = trial.suggest_int("POP_SIZE", 60, 160, step=20)
    max_gen = trial.suggest_int("MAX_GEN", 150, 450, step=50)
    patience = trial.suggest_int("PATIENCE", 30, 100, step=10)
    min_improvement = trial.suggest_float("MIN_IMPROVEMENT", 1e-4, 5e-3, log=True)
    mutation_rate = trial.suggest_float("MUTATION_RATE", 0.05, 0.35)
    elite_size = trial.suggest_int("ELITE_SIZE", 8, 25)
    tournament_k = trial.suggest_int("TOURNAMENT_K", 3, 7)

    score, dist_km, max_f, viol = run_ga_once(
        pop_size=pop_size,
        max_gen=max_gen,
        patience=patience,
        min_improvement=min_improvement,
        mutation_rate=mutation_rate,
        elite_size=elite_size,
        tournament_k=tournament_k,
        seed=123
    )

    # objective: minimize score, but punish violations more
    value = score + (viol * 50.0)

    trial.set_user_attr("dist_km", dist_km)
    trial.set_user_attr("max_fatigue", max_f)
    trial.set_user_attr("violations", viol)
    return value

print("\n" + "=" * 100)
print("🔧 OPTUNA TUNING START")
print("=" * 100)

study = optuna.create_study(direction="minimize")
study.optimize(optuna_objective, n_trials=25, show_progress_bar=True)

best_params = study.best_params

print("\n" + "=" * 100)
print("✅ OPTUNA BEST PARAMS")
print("=" * 100)
for k, v in best_params.items():
    print(f"{k}: {v}")

# save best params to file for report
params_path = os.path.join(OUT_DIR, "optuna_best_params.txt")
with open(params_path, "w", encoding="utf-8") as f:
    f.write("OPTUNA BEST PARAMS\n")
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
print(f"✓ Saved best params: {params_path}")

# apply best params
LAMBDA_FATIGUE = best_params["LAMBDA_FATIGUE"]
MU_VIOLATIONS = best_params["MU_VIOLATIONS"]
RECOVERY_FACTOR = best_params["RECOVERY_FACTOR"]
EXTREME_EFFORT_M = best_params["EXTREME_EFFORT_M"]

FINAL_POP = best_params["POP_SIZE"]
FINAL_MAX_GEN = best_params["MAX_GEN"]
FINAL_PATIENCE = best_params["PATIENCE"]
FINAL_MIN_IMPROVEMENT = best_params["MIN_IMPROVEMENT"]
FINAL_MUTATION = best_params["MUTATION_RATE"]
FINAL_ELITE = best_params["ELITE_SIZE"]
FINAL_TOURN = best_params["TOURNAMENT_K"]

# ==========================================
# 7) FINAL RUNS (pakai parameter Optuna)
# ==========================================
NUM_RUNS = 5
print("\n" + "=" * 100)
print(f"🚀 FINAL GA RUNS = {NUM_RUNS} (pakai Optuna-best params)")
print("=" * 100)

all_results = []
best_overall_route = None
best_overall_score = float("inf")

for run_num in range(1, NUM_RUNS + 1):
    print(f"\n📊 FINAL RUN {run_num}/{NUM_RUNS}")

    ga = GeneticAlgorithmFatigueTSP(
        pop_size=int(FINAL_POP),
        max_gen=int(FINAL_MAX_GEN),
        patience=int(FINAL_PATIENCE),
        min_improvement=float(FINAL_MIN_IMPROVEMENT),
        mutation_rate=float(FINAL_MUTATION),
        elite_size=int(FINAL_ELITE),
        tournament_k=int(FINAL_TOURN),
        run_name=f"Final Run {run_num}",
        seed=1000 + run_num
    )
    ga.evolve(verbose=True)

    route = ga.best_route
    score, dist_km, max_f, viol, f_hist = get_metrics(route)

    print(f"✅ Score={score:.2f} | Dist={dist_km:.2f}km | MaxFatigue={max_f:.2f} | Viol={viol} | Gen={ga.stopped_at_gen}")

    all_results.append({
        "Run": run_num,
        "Score": score,
        "Dist_km": dist_km,
        "MaxFatigue": max_f,
        "Violations": viol,
        "Generations": ga.stopped_at_gen,
        "Route": route.copy(),
        "ScoreHistory": ga.score_history.copy()
    })

    if score < best_overall_score:
        best_overall_score = score
        best_overall_route = route.copy()

# ==========================================
# 8) EXPORT SUMMARY + PLOTS
# ==========================================
df = pd.DataFrame([{
    "Run": r["Run"],
    "Score": r["Score"],
    "Dist_km": r["Dist_km"],
    "MaxFatigue": r["MaxFatigue"],
    "Violations": r["Violations"],
    "Generations": r["Generations"]
} for r in all_results])

summary_csv = os.path.join(OUT_DIR, "summary_results_ga_optuna.csv")
df.to_csv(summary_csv, index=False, encoding="utf-8")
print(f"\n✓ {summary_csv}")
print("\n" + df.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.plot(df["Run"], df["Score"], marker="o", linewidth=2)
plt.title("GA (Optuna-tuned) Score per Run (Lower is Better)")
plt.xlabel("Run")
plt.ylabel("Score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "score_per_run.png"), dpi=300)
plt.close()

best_score, best_dist, best_maxf, best_viol, best_f_hist = get_metrics(best_overall_route)
plt.figure(figsize=(12, 5))
plt.plot(best_f_hist, marker="o", linewidth=2, color="#EA4335")
plt.fill_between(range(len(best_f_hist)), best_f_hist, alpha=0.15, color="#EA4335")
plt.title("Fatigue Profile (Best Route - Optuna tuned)")
plt.xlabel("Step")
plt.ylabel("Fatigue")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fatigue_profile_best.png"), dpi=300)
plt.close()

best_run = min(all_results, key=lambda r: r["Score"])
plt.figure(figsize=(10, 5))
plt.plot(best_run["ScoreHistory"], linewidth=2, color="#005aab")
plt.title("Convergence (Best Run) - Score vs Generation")
plt.xlabel("Generation")
plt.ylabel("Score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "convergence_best_run.png"), dpi=300)
plt.close()

# ==========================================
# 9) FOLIUM MAP (OUTPUT HTML)
# ==========================================
print("\n🗺️ Creating Folium map...\n")

best_route = best_overall_route
score, dist_km, max_f, viol, f_hist = get_metrics(best_route)

best_time_minutes = dist_km / 20 * 60
hours = int(best_time_minutes // 60)
mins = int(best_time_minutes % 60)

center_lat = float(np.mean([c[0] for c in coords_values]))
center_lon = float(np.mean([c[1] for c in coords_values]))
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="CartoDB positron")

route_indices = best_route + [best_route[0]]
route_coords_ordered = [f"{coords_values[i][1]},{coords_values[i][0]}" for i in route_indices]
url_route = f"http://router.project-osrm.org/route/v1/bike/{';'.join(route_coords_ordered)}?overview=full&geometries=geojson"

try:
    res = requests.get(url_route, timeout=60).json()
    if res.get("code") == "Ok":
        geom = [(lat, lon) for lon, lat in res["routes"][0]["geometry"]["coordinates"]]
        plugins.AntPath(
            locations=geom,
            dash_array=[10, 20],
            delay=800,
            color="#005aab",
            pulse_color="white",
            weight=5,
            opacity=0.9,
        ).add_to(m)
except Exception as e:
    print(f"⚠ Route geometry error: {e}")

for idx, route_idx in enumerate(best_route):
    coord = coords_values[route_idx]
    elev = elevations_list[route_idx]
    next_idx = best_route[(idx + 1) % len(best_route)]
    segment_dist = distance_matrix[route_idx][next_idx]
    effort = max(0.0, elevations_list[next_idx] - elevations_list[route_idx])

    marker_text = " ⭐ START/END" if route_idx == start_location_idx else ""
    icon_color = "green" if route_idx == start_location_idx else "blue"

    popup_html = f"""
    <div style="font-family: Arial; width: 340px;">
        <b style="color: #005aab;">🚴 STOP {idx + 1}/{len(best_route)}</b><br>
        <b>{names[route_idx]}{marker_text}</b><br>
        <hr style="margin: 5px 0;">
        <b>🏔️ Elevation: {elev:.1f} m</b><br>
        <b>📏 Distance to next: {segment_dist:.2f} km</b><br>
        <b>⬆ Effort (climb): {effort:.1f} m</b>
    </div>
    """

    folium.Marker(
        location=coord,
        popup=folium.Popup(popup_html, max_width=340),
        icon=folium.Icon(color=icon_color, icon="bicycle", prefix="fa")
    ).add_to(m)

info_html = f"""
<div style="position: fixed; top: 20px; left: 70px; width: 560px;
            background-color: white; border-radius: 12px; z-index: 9999;
            font-family: 'Segoe UI'; box-shadow: 0 4px 16px rgba(0,0,0,0.15);">
<div style="display: flex; gap: 12px; padding: 20px; border-bottom: 1px solid #e0e0e0;
            background: linear-gradient(135deg, #005aab 0%, #003a70 100%);">
    <div style="font-size: 40px;">🚴</div>
    <div>
        <div style="font-size: 20px; font-weight: 700; color: white;">GA FATIGUE-AWARE ROUTE (Optuna tuned)</div>
        <div style="font-size: 12px; color: #ddd;">score = dist + λ·max_fatigue + μ·viol</div>
    </div>
</div>

<div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 0;">
    <div style="padding: 14px; text-align: center; border-right: 1px solid #f0f0f0;">
        <div style="font-size: 11px; color: #757575;">📍 JARAK</div>
        <div style="font-size: 18px; font-weight: 700;">{dist_km:.2f} km</div>
    </div>
    <div style="padding: 14px; text-align: center; border-right: 1px solid #f0f0f0;">
        <div style="font-size: 11px; color: #757575;">🔥 MAX FATIGUE</div>
        <div style="font-size: 18px; font-weight: 700; color: #FFA500;">{max_f:.2f}</div>
    </div>
    <div style="padding: 14px; text-align: center;">
        <div style="font-size: 11px; color: #757575;">⛔ VIOLATIONS</div>
        <div style="font-size: 18px; font-weight: 700; color: #EA4335;">{viol}</div>
    </div>
</div>

<div style="padding: 14px 20px; background-color: #f5f5f5; text-align: center; border-bottom: 1px solid #e0e0e0;">
    <div style="font-size: 12px;">🕐 WAKTU (20 km/h - SEPEDA)</div>
    <div style="font-size: 20px; font-weight: 700;">{best_time_minutes:.1f} min ({hours}h {mins}m)</div>
</div>

<div style="padding: 12px 20px; font-size: 12px; line-height: 1.8;">
    <div>🎯 Start/End: {START_CITY.split('. ')[1]}</div>
    <div>📍 Total Stops: {len(best_route)}</div>
    <div>⚙️ Algorithm: Genetic Algorithm (GA)</div>
    <div style="margin-top: 8px; padding-top: 8px; border-top: 1px solid #e0e0e0; color: #666;">
        λ={LAMBDA_FATIGUE:.3f}, μ={MU_VIOLATIONS:.3f}, r={RECOVERY_FACTOR:.3f}, extreme>{EXTREME_EFFORT_M:.2f}m
    </div>
</div>
</div>
"""
m.get_root().html.add_child(folium.Element(info_html))

out_html = os.path.join(OUT_DIR, "optimal_route_ga_fatigue_optuna_31.html")
m.save(out_html)
print(f"✓ {out_html}")

# ==========================================
# 10) ROUTE DETAIL CSV
# ==========================================
route_detail = []
fatigue = 0.0
for idx in range(len(best_route)):
    cur = best_route[idx]
    nxt = best_route[(idx + 1) % len(best_route)]

    seg_dist = distance_matrix[cur][nxt]
    elev_diff = elevations_list[nxt] - elevations_list[cur]
    effort = max(0.0, elev_diff)
    violation = 1 if effort > EXTREME_EFFORT_M else 0

    fatigue = (fatigue + effort) * RECOVERY_FACTOR

    marker = " ⭐ START/END" if cur == start_location_idx else ""
    route_detail.append({
        "Stop": idx + 1,
        "Location": names[cur] + marker,
        "Elevation_m": elevations_list[cur],
        "Dist_to_next_km": round(float(seg_dist), 3),
        "Effort_climb_m": round(float(effort), 1),
        "Violation": violation,
        "Fatigue_after_leg": round(float(fatigue), 3)
    })

df_route = pd.DataFrame(route_detail)
route_csv = os.path.join(OUT_DIR, "route_detail_ga_optuna_31.csv")
df_route.to_csv(route_csv, index=False, encoding="utf-8")
print(f"✓ {route_csv}")

# ==========================================
# 11) SUMMARY
# ==========================================
print("\n" + "=" * 100)
print("✅ OPTIMIZATION COMPLETE - GA + OPTUNA (31 lokasi)")
print("=" * 100)
print(f"""
BEST RESULT:
  Score       : {best_score:.2f}
  Distance    : {best_dist:.2f} km
  Max Fatigue : {best_maxf:.2f}
  Violations  : {best_viol}

Saved best params:
  • {params_path}

FILES:
  • {OUT_DIR}/optimal_route_ga_fatigue_optuna_31.html (🗺️ peta folium untuk screenshot)
  • {OUT_DIR}/optuna_best_params.txt                  (⚙️ parameter terbaik untuk laporan)
  • {OUT_DIR}/summary_results_ga_optuna.csv           (📋 ringkasan run)
  • {OUT_DIR}/route_detail_ga_optuna_31.csv           (📋 detail rute)
  • {OUT_DIR}/score_per_run.png                       (📈 skor per run)
  • {OUT_DIR}/fatigue_profile_best.png                (📈 fatigue terbaik)
  • {OUT_DIR}/convergence_best_run.png                (📈 konvergensi run terbaik)
""")

d:\Semester 6\SC\softcomputing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📡 Mengambil Distance Matrix untuk SEPEDA dari OSRM...



[I 2026-04-28 22:06:22,992] A new study created in memory with name: no-name-1672bddd-ac8b-4c90-b334-548338876e2b


✅ Distance Matrix OK! cached: hasil_ga_fatigue_optuna_31\distance_matrix_km.npy

🔧 OPTUNA TUNING START


  0%|          | 0/25 [00:00<?, ?it/s]C:\Users\asus\AppData\Local\Temp\ipykernel_26748\2254926748.py:216: RuntimeWarning: invalid value encountered in scalar divide
  improvement = (last_best - best_score_gen) / (last_best + 1e-12)
Best trial: 0. Best value: 180.797:   4%|▍         | 1/25 [00:01<00:32,  1.37s/it]

...

[I 2026-04-28 22:06:37,795] Trial 22 finished with value: 106.78399971776415 and parameters: {'LAMBDA_FATIGUE': 1.0905549867488673, 'MU_VIOLATIONS': 49.091685587283145, 'RECOVERY_FACTOR': 0.7990441073528979, 'EXTREME_EFFORT_M': 14.892068078087455, 'POP_SIZE': 100, 'MAX_GEN': 250, 'PATIENCE': 40, 'MIN_IMPROVEMENT': 0.0008892435938905403, 'MUTATION_RATE': 0.32471410751867374, 'ELITE_SIZE': 23, 'TOURNAMENT_K': 4}. Best is trial 8 with value: 96.8646339986243.


Best trial: 8. Best value: 96.8646:  96%|█████████▌| 24/25 [00:15<00:00,  2.46it/s]

[I 2026-04-28 22:06:38,180] Trial 23 finished with value: 178.21458158830654 and parameters: {'LAMBDA_FATIGUE': 1.0100273693792148, 'MU_VIOLATIONS': 27.365207839685173, 'RECOVERY_FACTOR': 0.7824654441165557, 'EXTREME_EFFORT_M': 12.888793067013836, 'POP_SIZE': 80, 'MAX_GEN': 200, 'PATIENCE': 50, 'MIN_IMPROVEMENT': 0.0022098114874820065, 'MUTATION_RATE': 0.21650452025929465, 'ELITE_SIZE': 20, 'TOURNAMENT_K': 4}. Best is trial 8 with value: 96.8646339986243.


Best trial: 8. Best value: 96.8646: 100%|██████████| 25/25 [00:15<00:00,  1.60it/s]


[I 2026-04-28 22:06:38,753] Trial 24 finished with value: 169.45588398286128 and parameters: {'LAMBDA_FATIGUE': 0.6285896918332967, 'MU_VIOLATIONS': 12.558805890730328, 'RECOVERY_FACTOR': 0.7529230212569814, 'EXTREME_EFFORT_M': 13.324812392982075, 'POP_SIZE': 100, 'MAX_GEN': 300, 'PATIENCE': 60, 'MIN_IMPROVEMENT': 0.0010647313565307853, 'MUTATION_RATE': 0.24953083089451197, 'ELITE_SIZE': 23, 'TOURNAMENT_K': 3}. Best is trial 8 with value: 96.8646339986243.

✅ OPTUNA BEST PARAMS
LAMBDA_FATIGUE: 0.8276151553903512
MU_VIOLATIONS: 24.722793236327053
RECOVERY_FACTOR: 0.7182883065198782
EXTREME_EFFORT_M: 14.174807692127592
POP_SIZE: 80
MAX_GEN: 250
PATIENCE: 60
MIN_IMPROVEMENT: 0.001427470342228902
MUTATION_RATE: 0.1619801202953678
ELITE_SIZE: 21
TOURNAMENT_K: 4
✓ Saved best params: hasil_ga_fatigue_optuna_31\optuna_best_params.txt

🚀 FINAL GA RUNS = 5 (pakai Optuna-best params)

📊 FINAL RUN 1/5
🧬 Final Run 1...

  Gen  50: score=104.25 | dist=92.74km | maxF=13.90 | viol=0
🛑 Early stop at ge